# Preparation

In [1]:
import numpy as np
import os
import glob
import librosa
import librosa.display
import pandas as pd
import random
import math
import matplotlib.pyplot as plt
from keras.models import Sequential, load_model, Model
from tensorflow.keras.utils import Sequence
from keras.utils import np_utils
from tqdm import tqdm
import seaborn as sn
from sklearn import model_selection
from sklearn import preprocessing
import IPython.display as ipd
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, UpSampling2D, concatenate

In [2]:
import tensorflow as tf
print(tf.__version__)

import keras
print(keras.__version__)

2.12.0
2.12.0


In [3]:
# define directories
base_dir = "/content/"
meta_file = os.path.join(base_dir, "meta_10sec_gcp.csv")

In [4]:
meta_data = pd.read_csv(meta_file)
meta_data

,id,audio_path,label_path
0,Br2-1-0,/content/cut10/Br2-1Cut/Br2-1Cut0.wav,/content/cut10/Br2-1VnCut/Br2-1VnCut0.wav
1,Br2-1-1,/content/cut10/Br2-1Cut/Br2-1Cut1.wav,/content/cut10/Br2-1VnCut/Br2-1VnCut1.wav
2,Br2-1-2,/content/cut10/Br2-1Cut/Br2-1Cut2.wav,/content/cut10/Br2-1VnCut/Br2-1VnCut2.wav
3,Br2-1-3,/content/cut10/Br2-1Cut/Br2-1Cut3.wav,/content/cut10/Br2-1VnCut/Br2-1VnCut3.wav
4,Br2-1-4,/content/cut10/Br2-1Cut/Br2-1Cut4.wav,/content/cut10/Br2-1VnCut/Br2-1VnCut4.wav
...,...,...,...
265,Br2-4-55,/content/cut10/Br2-4Cut/Br2-4Cut55.wav,/content/cut10/Br2-4VnCut/Br2-4VnCut55.wav
266,Br2-4-56,/content/cut10/Br2-4Cut/Br2-4Cut56.wav,/content/cut10/Br2-4VnCut/Br2-4VnCut56.wav
267,Br2-4-57,/content/cut10/Br2-4Cut/Br2-4Cut57.wav,/content/cut10/Br2-4VnCut/Br2-4VnCut57.wav
268,Br2-4-58,/content/cut10/Br2-4Cut/Br2-4Cut58.wav,/content/cut10/Br2-4VnCut/Br2-4VnCut58.wav


In [ ]:
#!curl -sc /tmp/cookie "https://drive.google.com/uc?export=download&id=1Zrjv7dCHSWE8amwYw7AnGowHM5GQREH-" > /dev/null
#!CODE="$(awk '/_warning_/ {print $NF}' /tmp/cookie)"
#!curl -Lb /tmp/cookie "https://drive.google.com/uc?export=download&confirm=${CODE}&id=1Zrjv7dCHSWE8amwYw7AnGowHM5GQREH-" -o /content/cut10.zip

In [ ]:
#!wget "https://drive.google.com/uc?export=download&id=1Zu8F-yRN-MCU9PXTHDacz4zWashjaMEN" -O meta_10sec_gcp.csv

In [ ]:
#!unzip /content/cut10.zip

# definitions

In [5]:
# load a wave data
def load_wave_data(file_path):
    sr = 16000
    x, fs = librosa.load(file_path, sr=sr)
    return x,fs

In [6]:
def cal_stft(audio_wav, label_wav, n_fft=1024, hop_length=128):
  audio_stft = np.abs(librosa.stft(audio_wav, n_fft=n_fft, hop_length=hop_length)).astype(np.float32)
  label_stft = np.abs(librosa.stft(label_wav, n_fft=n_fft, hop_length=hop_length)).astype(np.float32)
  #正規化
  norm = audio_stft.max()
  audio_stft /= norm
  label_stft /= norm
  return audio_stft, label_stft

In [7]:
# display wave in plots
def show_wave(x):
    plt.plot(x)
    plt.show()
# display wave in heatmap
def show_stft(stft, fs):
    librosa.display.specshow(stft, sr=fs)
    plt.colorbar()
    plt.show()

# スペクトログラムへの変換・保存・読み込み

In [ ]:
#!curl -sc /tmp/cookie "https://drive.google.com/uc?export=download&id=1YoxGWBSgSV1WW1Omhq4sarDsLJbF_wrG" > /dev/null
#!CODE="$(awk '/_warning_/ {print $NF}' /tmp/cookie)"
#!curl -Lb /tmp/cookie "https://drive.google.com/uc?export=download&confirm=${CODE}&id=1YoxGWBSgSV1WW1Omhq4sarDsLJbF_wrG" -o /content/ex3.npz

In [8]:
train_data = np.load("/content/ex3.npz")

In [9]:
x_train = train_data["x"]
y_train = train_data["y"]

In [10]:
x_train = x_train.reshape(x_train.shape[0],x_train.shape[1],x_train.shape[2],1)
y_train = y_train.reshape(y_train.shape[0],y_train.shape[1],y_train.shape[2],1)

In [ ]:
x_train.shape

(270, 257, 834, 1)

In [ ]:
y_train.shape

(270, 257, 834, 1)

# モデル定義

In [11]:
from keras.models import Model, model_from_json
from keras.layers import Input, Conv2D, Conv2DTranspose, LeakyReLU, ReLU
from tensorflow.keras.layers import BatchNormalization, Dropout, concatenate
from keras.optimizers import Adam

In [12]:
from keras import backend

In [13]:
def adj_concat(base, target):
    base_h = base.shape[1]
    base_w = base.shape[2]
    target_h = target.shape[1]
    target_w = target.shape[2]

    # shapeの上下方向と左右方向の差分を確認
    diff_h = base_h - target_h
    diff_w = base_w - target_w

    # 上下方向
    if diff_h != 0:
        pad_h = abs(diff_h)
        # 差分が偶数→上下ゼロパディング
        if (diff_h % 2) == 0:
            # baseの方が大きい→targetを調整
            if diff_h > 0:
                target = keras.layers.ZeroPadding2D(padding=(int(pad_h/2), 0))(target)
            # targetの方が大きい→baseを調整
            else:
                base = keras.layers.ZeroPadding2D(padding=(int(pad_h/2), 0))(base)
        # 差分が奇数→上側だけゼロパディング
        else:
            # baseの方が大きい→targetを調整
            if diff_h > 0:
                for i in range(pad_h):
                    if i % 2:
                        target = keras.layers.ZeroPadding2D(padding=((1, 0), (0, 0)))(target)
                    else:
                        target = keras.layers.ZeroPadding2D(padding=((0, 1), (0, 0)))(target)
            # targetの方が大きい→baseを調整
            else:
                for i in range(pad_h):
                    if i % 2:
                        base = keras.layers.ZeroPadding2D(padding=((1, 0), (0, 0)))(base)
                    else:
                        base = keras.layers.ZeroPadding2D(padding=((0, 1), (0, 0)))(base)
    # 左右方向
    if diff_w != 0:
        pad_w = abs(diff_w)
        # 差分が偶数→左右ゼロパディング
        if (diff_w % 2) == 0:
            # baseの方が大きい→targetを調整
            if diff_w > 0:
                target = keras.layers.ZeroPadding2D(padding=(0, int(pad_w/2)))(target)
            # targetの方が大きい→baseを調整
            else:
                base = keras.layers.ZeroPadding2D(padding=(0, int(pad_w/2)))(base)
        # 差分が奇数→左側だけゼロパディング
        else:
            # baseの方が大きい→targetを調整
            if diff_w > 0:
                for i in range(pad_w):
                    if i % 2:
                        target = keras.layers.ZeroPadding2D(padding=((0, 0), (1, 0)))(target)
                    else:
                        target = keras.layers.ZeroPadding2D(padding=((0, 0), (0, 1)))(target)
            # targetの方が大きい→baseを調整
            else:
                for i in range(pad_w):
                    if i % 2:
                        base = keras.layers.ZeroPadding2D(padding=((0, 0), (1, 0)))(base)
                    else:
                        base = keras.layers.ZeroPadding2D(padding=((0, 0), (0, 1)))(base)

    return base, target

In [14]:
def LSD_loss(y_true, y_pred):
    LSD = backend.mean((y_true - y_pred)**2, axis=2)
    LSD = backend.mean(backend.sqrt(LSD), axis=1)
    return LSD


#Construct the U-Net model with Functional API by Keras
def unet(input_shape):
  inputs = Input(input_shape)

  enc1 = Conv2D(32, kernel_size=(5, 7), padding='same')(inputs)
  enc1 = BatchNormalization()(enc1)
  enc1 = LeakyReLU(alpha=0.2)(enc1)

  enc2 = Conv2D(64, kernel_size=(5, 7), padding='same')(enc1)
  enc2 = BatchNormalization()(enc2)
  enc2 = LeakyReLU(alpha=0.2)(enc2)

  enc3 = Conv2D(128, kernel_size=(5, 7), padding='same')(enc2)
  enc3 = BatchNormalization()(enc3)
  enc3 = LeakyReLU(alpha=0.2)(enc3)

  enc4 = Conv2D(256, kernel_size=(5, 5), padding='same')(enc3)
  enc4 = BatchNormalization()(enc4)
  enc4 = LeakyReLU(alpha=0.2)(enc4)

  enc5 = Conv2D(256, kernel_size=(5, 5), padding='same')(enc4)
  enc5 = BatchNormalization()(enc5)
  enc5 = LeakyReLU(alpha=0.2)(enc5)

  enc6 = Conv2D(256, kernel_size=(3, 3), padding='same')(enc5)
  enc6 = BatchNormalization()(enc6)
  enc6 = LeakyReLU(alpha=0.2)(enc6)

  enc7 = Conv2D(256, kernel_size=(3, 3), padding='same')(enc6)
  enc7 = BatchNormalization()(enc7)
  enc7 = LeakyReLU(alpha=0.2)(enc7)

  enc8 = Conv2D(256, kernel_size=(3, 3), padding='same')(enc7)
  enc8 = BatchNormalization()(enc8)
  enc8 = LeakyReLU(alpha=0.2)(enc8)

  dec1 = Conv2DTranspose(256, kernel_size=(3, 3), padding='same')(enc8)
  dec1 = BatchNormalization()(dec1)
  dec1 = ReLU()(dec1)
  dec1 = Dropout(0.5)(dec1)

  dec1, enc7 = adj_concat(dec1, enc7)
  dec2 = concatenate([dec1, enc7], axis=-1)
  dec2 = Conv2DTranspose(256, kernel_size=(3, 3), padding='same', kernel_initializer = "he_uniform")(dec2)
  dec2 = BatchNormalization()(dec2)
  dec2 = ReLU()(dec2)
  dec2 = Dropout(0.5)(dec2)

  dec2, enc6 = adj_concat(dec2, enc6)
  dec3 = concatenate([dec2, enc6], axis=-1)
  dec3 = Conv2DTranspose(256, kernel_size=(3, 3), padding='same', kernel_initializer = "he_uniform")(dec3)
  dec3 = BatchNormalization()(dec3)
  dec3 = ReLU()(dec3)
  dec3 = Dropout(0.5)(dec3)

  dec3, enc5 = adj_concat(dec3, enc5)
  dec4 = concatenate([dec3, enc5], axis=-1)
  dec4 = Conv2DTranspose(256, kernel_size=(5, 5), padding='same', kernel_initializer = "he_uniform")(dec4)
  dec4 = BatchNormalization()(dec4)
  dec4 = ReLU()(dec4)

  dec4, enc4 = adj_concat(dec4, enc4)
  dec5 = concatenate([dec4, enc4], axis=-1)
  dec5 = Conv2DTranspose(128, kernel_size=(5, 5), padding='same', kernel_initializer = "he_uniform")(dec5)
  dec5 = BatchNormalization()(dec5)
  dec5 = ReLU()(dec5)

  dec5, enc3 = adj_concat(dec5, enc3)
  dec6 = concatenate([dec5, enc3], axis=-1)
  dec6 = Conv2DTranspose(64, kernel_size=(5, 7), padding='same', kernel_initializer = "he_uniform")(dec6)
  dec6 = BatchNormalization()(dec6)
  dec6 = ReLU()(dec6)

  dec6, enc2 = adj_concat(dec6, enc2)
  dec7 = concatenate([dec6, enc2], axis=-1)
  dec7 = Conv2DTranspose(32, kernel_size=(5, 7), padding='same', kernel_initializer = "he_uniform")(dec7)
  dec7 = BatchNormalization()(dec7)
  dec7 = ReLU()(dec7)

  dec7, enc1 = adj_concat(dec7, enc1)
  dec8 = concatenate([dec7, enc1], axis=-1)
  dec8 = Conv2DTranspose(1, kernel_size=(5, 7), padding='same', activation='sigmoid')(dec8)


  model = Model(inputs=inputs, outputs=dec8)
  return model

# モデルの構築
#input_shape = x_train.shape[1:]
input_shape = (257,834,1)
model = unet(input_shape)

# モデルのコンパイル
model.compile(optimizer=Adam(lr= 0.0001), loss='mean_absolute_error', metrics=['accuracy'])

# モデルのサマリーを表示
model.summary()





Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 257, 834, 1  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 257, 834, 32  1152        ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 257, 834, 32  128        ['conv2d[0][0]']                 
 alization)                     )                                                             

/usr/local/lib/python3.10/dist-packages/keras/optimizers/legacy/adam.py:117: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


                                )                                                                 
                                                                                                  
 concatenate_5 (Concatenate)    (None, 257, 834, 12  0           ['re_lu_5[0][0]',                
                                8)                                'leaky_re_lu_1[0][0]']          
                                                                                                  
 conv2d_transpose_6 (Conv2DTran  (None, 257, 834, 32  143392     ['concatenate_5[0][0]']          
 spose)                         )                                                                 
                                                                                                  
 batch_normalization_14 (BatchN  (None, 257, 834, 32  128        ['conv2d_transpose_6[0][0]']     
 ormalization)                  )                                                                 
          

# 新しいセクション

In [16]:
batch_size = 1
num_epochs = 5

history = model.fit(x_train, y_train, batch_size=batch_size, epochs=num_epochs, validation_split=0.1)

Epoch 1/5
243/243 [==============================] - 372s 1s/step - loss: 0.0023 - accuracy: 0.2585 - val_loss: 0.0031 - val_accuracy: 0.0424
Epoch 2/5
243/243 [==============================] - 349s 1s/step - loss: 0.0023 - accuracy: 0.2585 - val_loss: 0.0030 - val_accuracy: 0.0424
Epoch 3/5
243/243 [==============================] - 349s 1s/step - loss: 0.0023 - accuracy: 0.2585 - val_loss: 0.0030 - val_accuracy: 0.0424
Epoch 4/5
243/243 [==============================] - 349s 1s/step - loss: 0.0022 - accuracy: 0.2585 - val_loss: 0.0029 - val_accuracy: 0.0424
Epoch 5/5
243/243 [==============================] - 349s 1s/step - loss: 0.0022 - accuracy: 0.2585 - val_loss: 0.0029 - val_accuracy: 0.0424


In [17]:
model.save('model2.h5')

In [15]:
model.load_weights('model.h5')

In [19]:
samplename = "x.wav"
sample_x, fs_sample_x = load_wave_data(samplename)
samx_stft = np.abs(librosa.stft(sample_x, n_fft=512, hop_length=128)).astype(np.float32)
sam_norm = samx_stft.max()
samx_stft /= sam_norm

x_sample = samx_stft.reshape(1,samx_stft.shape[0],samx_stft.shape[1],1)

y_sample = model.predict(x_sample)
y_sample *= sam_norm
y_sample = y_sample.reshape(y_sample.shape[1],y_sample.shape[2])
from librosa.core import istft, resample
import soundfile as sf
phase = np.exp(1.j*np.angle(librosa.stft(sample_x, n_fft=512, hop_length=128)))
sample_y = istft(y_sample*phase, hop_length=128, win_length=512)
sf.write("y_predict.wav", sample_y, fs_sample_x)

1/1 [==============================] - 9s 9s/step


In [ ]:
val_ratio = 0.1
number = int(meta_data.shape[0]*(1-val_ratio))
meta_train = meta_data[0:number]
meta_val = meta_data[number:]
meta_val = meta_val.rename(index=lambda s: s-number)

In [ ]:
n=834

#generator
def gen(meta, batch_size=32):
  k=0
  while True:
    batch_x = np.zeros((batch_size, 257, n), np.float32)
    batch_y = np.zeros((batch_size, 257, n), np.float32)
    for i in range((k+1)*batch_size):
      x, fs_x = load_wave_data(meta.loc[k*batch_size+i,"audio_path"])
      y, fs_y = load_wave_data(meta.loc[k*batch_size+i,"label_path"])
      fs = fs_x
      a_stft, l_stft = cal_stft(x,y, n_fft=512)
      batch_x[i] = a_stft
      batch_y[i] = l_stft
    batch_x = batch_x.reshape(batch_x.shape[0],batch_x.shape[1],batch_x.shape[2],1)
    batch_y = batch_y.reshape(batch_y.shape[0],batch_y.shape[1],batch_y.shape[2],1)
    if (i + 1) * batch_size >= len(x):
      k=0
    else:
      k+=1
    yield batch_x, batch_y


In [ ]:
batch_size = 1
len_train = number
len_valid = meta_data.shape[0]-number

model.fit_generator(
    gen(meta_train, batch_size=batch_size),
    validation_data=gen(meta_val, batch_size=batch_size),
    steps_per_epoch=len_train // batch_size,
    validation_steps=len_valid // batch_size,
    max_queue_size=2
)